I was not careful with the intialization of the network in this video. (1) What is the loss you'd get if the predicted probabilities at initialization were perfectly uniform? What loss do we achieve? (2) Can you tune the initialization to get a starting loss that is much more similar to (1)?


In [5]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline


In [6]:
words = open('names.txt','r').read().splitlines()

In [7]:
len(words)

32033

In [8]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [9]:
block_size = 3
def build_dataset(words):
  x = []
  y = []
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      x.append(context)
      y.append(ix)
      context = context[1:] + [ix]
  x = torch.tensor(x)
  y = torch.tensor(y)
  print(x.shape,y.shape)
  return x , y

import random
random.seed(42)
random.shuffle(words)

n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

xtr,ytr = build_dataset(words[:n1])
xdev,ydev = build_dataset(words[n1:n2])
xte,yte = build_dataset(words[n2:])







torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [10]:
# Loss if all probs are equally likely
probs = torch.tensor(1/len(stoi))
loss = -probs.log()
loss

tensor(3.2958)

In [19]:
n_emb = 100
n_hidden = 100
g = torch.Generator().manual_seed(2147483647)

c = torch.randn((27, n_emb), generator=g)
w1 = torch.randn((block_size * n_emb, n_hidden), generator=g)
b1 = torch.randn(n_hidden, generator=g)

# Output layer modifications
w2 = torch.randn((n_hidden, 27), generator=g) * 0.01
b2 = torch.zeros(27)  # Zeroing out bias gives uniform logits (all ~0)

parameters = [c, w1, b1, w2, b2]

minibatch = torch.randint(0, xtr.shape[0], (32,))
emb = c[xtr[minibatch]].view(-1, n_emb * 3)
l1 = torch.tanh(emb @ w1 + b1)
logits = l1 @ w2 + b2

loss_train = F.cross_entropy(logits, ytr[minibatch])
print(f"Loss at first pass : {round(loss_train.item(), 3)}")
# Output: Loss at first pass : 3.296 (or ~3.30)



Loss at first pass : 3.294
